<a href="https://colab.research.google.com/github/vad-source/DRL/blob/main/RAG/NaiveRAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## NLP APPLICATIONS
**Designed by:** RAJA VADHANA PRABHAKAR  
**Organization:** BITS PILANI WILP  
**Purpose:** Academic Training / Proof of Concept  

---
#### Attribution & AI Disclosure
- **Original Design:** The logic, architecture, and modular structure of this notebook were designed by the author.
- **Development Assistance:** Generative AI (e.g., ChatGPT/Claude/Copilot) was used for coding implementation and debugging support.
- **License:** This work is licensed under the [Apache License 2.0](https://apache.org).

# Naive RAG

In [ ]:
!pip install faiss-cpu sentence-transformers

## Knowledge Base
> Documents | Knowledge Graph | Databases

In [ ]:
documents = [
    "Refunds are processed within 3 to 5 working days as per RBI guidelines",
    "If refund is delayed beyond 5 days, escalation can be initiated",
    "Customers should not be promised instant refunds",
    "Transaction issues can be checked using transaction ID",
    "Refund status queries fall under billing support"
]

POLICIES = [
    "Refunds take 3-5 business days",
    "UPI reversals take 24 hours",
    "Failed transactions auto-reverse"
]

## Ingestion & Preprocessor
> Natural Language Understanding

> Query Embedding

> > OpenAI / sentence-transformers, Hugging Face , Customized Code


In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# Lightweight embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
# Create embeddings
doc_embeddings = model.encode(documents)

# Convert to float32 (FAISS requirement)
doc_embeddings = np.array(doc_embeddings).astype("float32")

# Create FAISS index
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)

# Add vectors
index.add(doc_embeddings)

print("FAISS index built with", index.ntotal, "documents")
print("\n\tSample one of the input document : \" ",documents[1], "\"")
print("\n\tSize of the embedding returned for this document : ",doc_embeddings[1].size)
print("\n\tSample first three features : ",doc_embeddings[1][0], ":" , doc_embeddings[1][1] ,":", doc_embeddings[1][2])
print("\n\t",index)

### Helper Functions : GEC | Entity Protection | etc.,

In [ ]:
# --- SETUP ---
import re
from typing import List, Dict, Tuple

# Hardcoded slang dictionary (toy)
SLANG_MAP = {
    "dint": "didn't",
    "didnt": "didn't",
    "ordrr": "order",
    "ordr": "order",
    "2day": "today",
    "pls": "please",
    "plz": "please",
    "recvd": "received",
    "thx": "thanks"
}

# Regex for protected entities (txn IDs, amounts)
TXN_PATTERN = r"\b\d{5,}\b"  # simplistic txn id (>=5 digits)
#AMOUNT_PATTERN = r"\b\d+(\.\d{1,2})?\b"
AMOUNT_PATTERN = r"\b\d+(?:\.\d{1,2})?\b"

def extract_entities(text: str) -> Dict:
    txn_ids = re.findall(TXN_PATTERN, text)
    amounts = re.findall(AMOUNT_PATTERN, text)
    return {
        "txn_ids": txn_ids,
        "amounts": [a[0] if isinstance(a, tuple) else a for a in amounts]
    }

def protect_entities(text: str) -> Tuple[str, Dict]:
    entities = extract_entities(text)
    protected = text

    for i, txn in enumerate(entities["txn_ids"]):
        protected = protected.replace(txn, f"__TXN_{i}__")

    for i, amt in enumerate(entities["amounts"]):
        protected = protected.replace(amt, f"__AMT_{i}__")

    return protected, entities

def restore_entities(text: str, entities: Dict) -> str:
    restored = text
    for i, txn in enumerate(entities["txn_ids"]):
        restored = restored.replace(f"__TXN_{i}__", txn)
    for i, amt in enumerate(entities["amounts"]):
        restored = restored.replace(f"__AMT_{i}__", amt)
    return restored

def simple_gec(text: str):
    protected_text, entities = protect_entities(text)
    import re
    tokens = re.findall(r"\w+|[^\w\s]", protected_text)

    #tokens = protected_text.split()
    edits = []

    corrected_tokens = []
    for token in tokens:
        if token.lower() in SLANG_MAP:
            corrected = SLANG_MAP[token.lower()]
            edits.append({"from": token, "to": corrected})
            corrected_tokens.append(corrected)
        else:
            corrected_tokens.append(token)

    corrected = " ".join(corrected_tokens)
    corrected = corrected.capitalize()

    final_text = restore_entities(corrected, entities)

    confidence = 1 - (len(edits) * 0.1)  # naive confidence

    return {
        "corrected_text": final_text,
        "entities": entities,
        "edits": edits,
        "confidence": max(confidence, 0.5)
    }
def simple_gec_N(text: str):
    protected_text, entities = protect_entities(text)

    tokens = protected_text.split()
    edits = []
    corrected_tokens = []

    for i, token in enumerate(tokens):

        # skip protected placeholders
        if token.startswith("<") and token.endswith(">"):
            corrected_tokens.append(token)
            continue

        lower = token.lower()

        if lower in SLANG_MAP:
            corrected = SLANG_MAP[lower]

            edits.append({
                "index": i,
                "from": token,
                "to": corrected
            })

            corrected_tokens.append(corrected)
        else:
            corrected_tokens.append(token)

    corrected = " ".join(corrected_tokens)

    final_text = restore_entities(corrected, entities)

    confidence = max(1 - len(edits) * 0.1, 0.5)

    return {
        "corrected_text": final_text,
        "entities": entities,
        "edits": edits,
        "confidence": confidence
    }

# --- TEST ---
user_input = "i dint get my refund 2day for txn id 88392 pls chk"
gec_output = simple_gec(user_input)
print("GEC INPUT:\n", user_input)
print("GEC OUTPUT:\n", gec_output)

## Retriever
> Single path Semantic Search or Orchested Search

> > Vector Search | Graph Traversal(Query) | ExternalAPI

> > Vector Database ((Pinecone / Weaviate / FAISS) |  Graph Database (Neo4J) Cypher / Gremlin | weather/ Stock | Customized Code


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer()
doc_vectors = vectorizer.fit_transform(documents)

def naive_rag(query: str, top_k=2):
    query_vec = vectorizer.transform([query])
    scores = cosine_similarity(query_vec, doc_vectors)[0]
    top_indices = scores.argsort()[-top_k:][::-1]

    results = [{"doc": documents[i], "score": scores[i]} for i in top_indices]
    return results

# --- TEST ---
rag_results = naive_rag(gec_output["corrected_text"])
print("\nRAG RESULTS:\n", rag_results)

In [ ]:
def faiss_rag(query, top_k=2):
    query_embedding = model.encode([query]).astype("float32")

    distances, indices = index.search(query_embedding, top_k)

    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            "doc": documents[idx],
            "score": float(1 / (1 + distances[0][i]))  # convert distance → similarity
        })

    return results


# --- TEST ---
query = "I didn't get my refund today for transaction ID 88392"
rag_results = faiss_rag(query)

print("Sample RAG RESULTS:\n", rag_results)

## Generator
> LM | LLM | SLM
> Open AI GPT,LangChain, Meta Llama/LlamaIndex , Mistral AI, Customised


## Orchestrator
> In hybrid system this works immediately
> > after the preprocessing
> > > using classifier (LM) to determine the most appropriate RAG to call
> > > to trigger multisource retreival

> > after the response generation
> > > to Combine results --> Reason --> Iterate (if needed)

> Microsoft Orchestration | Customized  

In [ ]:
def simple_intent_classifier(text: str):
    if "refund" in text.lower():
        return {"intent": "refund_query", "confidence": 0.9}
    return {"intent": "unknown", "confidence": 0.6}

#### Hardcoded Template for customized responses

In [ ]:
def generate_response_old(gec, intent, rag_docs):
    context = " ".join([d["doc"] for d in rag_docs])

    response = f"""
I understand your concern.

Based on our policy: {context}

Let me check your transaction ID {gec['entities']['txn_ids']} and assist further.
"""
    return response.strip()


intent = simple_intent_classifier(gec_output["corrected_text"])
response = generate_response_old(gec_output, intent, rag_results)

print("\nFINAL RESPONSE:\n", response)

#### Language Model for responses

In [ ]:
!pip install transformers accelerate torch

In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="microsoft/Phi-3-mini-4k-instruct",
    device_map="auto"
)

def generate_response(gec, intent, rag_docs):

    context = "\n".join([d["doc"] for d in rag_docs])

    prompt = f"""
    We are a banking customer support assistant.

    Rules:
    - Be concise
    - Do not hallucinate
    - Use only provided policy
    - Mention txn id if available

    Customer Intent:
    {intent}

    Customer Message:
    {gec['corrected_text']}

    Relevant Policy:
    {context}

    Transaction IDs:
    {gec['entities']['txn_ids']}

    Respond professionally and briefly.
    """

    output = generator(
        prompt,
        max_new_tokens=120,
        do_sample=False
    )

    return output[0]["generated_text"]


intent = simple_intent_classifier(gec_output["corrected_text"])
response = generate_response(gec_output, intent, rag_results)

print("\nFINAL RESPONSE:\n", response)

## Evaluator
> Detect Failure:
> > Decide which layer need this : Confidence calibration across GEC → NLU → RAG → Response

> > Entity Corruption | Hallucination | Policy Violation

> > txn mismatch | response not grounded in retrieved docs | “instant refund” promise

> Score Confidence level based on these

In [ ]:
def detect_failure_modes(user_input, gec_output, response, rag_docs):
    failures = {}

    # Entity corruption
    original_txns = extract_entities(user_input)["txn_ids"]
    response_txns = extract_entities(response)["txn_ids"]
    failures["entity_corruption"] = original_txns != response_txns

    # Hallucination (semantic check)
    rag_text = " ".join([d["doc"] for d in rag_docs]).lower()
    response_text = response.lower()

    #failures["hallucination"] = not any(doc["doc"].lower() in response_text for doc in rag_docs )
    for sentence in response.lower().split("."):
        if sentence.strip() and sentence not in rag_text:
            hallucination = True
    failures["hallucination"] = hallucination

    # Policy violation
    failures["policy_violation"] = "instant refund" in response_text

    return failures

#---------------TEST
#failures = detect_failure_modes(user_input, gec_output, response, rag_results)
#print("\nFAILURE ANALYSIS:\n", failures)

In [ ]:
def calibrate_confidence(gec_conf, intent_conf, rag_results, failures):
    rag_conf = np.mean([r["score"] for r in rag_results])

    final_conf = (
        0.3 * gec_conf +
        0.3 * intent_conf +
        0.4 * rag_conf
    )

    # Penalties
    penalties = 0
    if failures["entity_corruption"]:
        penalties += 0.3
    if failures["hallucination"]:
        penalties += 0.2
    if failures["policy_violation"]:
        penalties += 0.4

    final_conf -= penalties

    return max(min(final_conf, 1.0), 0.0)

## Human In the Loop Trigger (if needed)

In [ ]:
def human_review_required(confidence, failures):
    if confidence < 0.6:
        return True
    if any(failures.values()):
        return True
    return False


In [ ]:
#rag_results = faiss_rag(gec_output["corrected_text"])

In [ ]:
user_input = "i dint get my refund 2day for txn id 88392 pls chk"

# Step 1: GEC
gec_output = simple_gec(user_input)

# Step 2: Intent
intent = simple_intent_classifier(gec_output["corrected_text"])

# Step 3: RAG (FAISS)
rag_results = faiss_rag(gec_output["corrected_text"])

# Step 4: Generate response
response = generate_response(gec_output, intent, rag_results)

# Step 5: Failure detection
failures = detect_failure_modes(user_input, gec_output, response, rag_results)

# Step 6: Confidence
final_conf = calibrate_confidence(
    gec_output["confidence"],
    intent["confidence"],
    rag_results,
    failures
)

# Step 7: Human review
review_flag = human_review_required(final_conf, failures)

print("\n--- FINAL OUTPUT ---")
print("User Input:", user_input)
print("GEC (Simple Rule based only) Corrected:\n\t", gec_output["corrected_text"])
print("\n\tResponse:\n", response)
print("\n\tFailures:", failures)
print("\n\tConfidence:", final_conf)
print("\n\tHuman Review:", review_flag)